<p align="center">
  <img src="https://img.shields.io/badge/Research%20Mode-ON-4cbb17?style=for-the-badge" alt="Research Mode">
</p>



# Loading Spatial Data in Python: Visium and GeoMx Basics

## Overview

Spatial transcriptomics lets us measure gene expression while keeping track of **where** each measurement came from in the tissue. This is a major advance over single-cell RNA-seq, where spatial information is lost during dissociation.

This tutorial covers two complementary spatial profiling technologies:

| Technology | Description | Resolution |
|---|---|---|
| **10x Visium** | Whole-transcriptome capture on a grid of ~5,000 barcoded spots | ~55 µm per spot |
| **NanoString GeoMx** | Targeted or whole-transcriptome profiling of manually selected regions of interest (ROIs) | User-defined ROIs |

### What you will learn
- How to load Visium and GeoMx data into Python as `AnnData` objects
- How to check that spatial coordinates are present and correctly aligned
- How to make basic diagnostic and exploratory plots
- How to save a clean `.h5ad` file for downstream analysis

### What this tutorial does NOT cover
Full preprocessing, batch correction, clustering, or differential expression — those come in later notebooks.

---

> **New to AnnData?** An `AnnData` object is the standard data container for single-cell and spatial data in Python. Think of it as a spreadsheet where rows = spots/cells and columns = genes, with extra slots for metadata, embeddings, and images. See the [AnnData docs](https://anndata.readthedocs.io) for a full overview.

# Table of Contents
1. [Setup](#Setup)    
2. [Set Dataset Paths](#set-dataset-paths)
3. [Explore Data](#explore-data)
    - Part A: Visium Data
    - Part B: GeoMx Data
4. [Save Outputs](#save-outputs)
5. [Next Steps](#next-steps)
6. [Provenance](#provenance)  

## 1. Setup

We install and import the core packages needed for this tutorial.

| Package | Purpose |
|---|---|
| `scanpy` | Single-cell / spatial analysis (load, preprocess, visualize) |
| `squidpy` | Spatial-specific analysis and plotting built on top of scanpy |
| `anndata` | The core data container format |
| `pandas` / `numpy` | General data manipulation |
| `matplotlib` | Plotting backend |

In [ ]:
import sys, subprocess, importlib, warnings, math, os, pandas as pd
from pathlib import Path

def setup_environment():
    """Configure logging, warnings, pandas options, and auto-install core packages."""

    def import_or_install(pkg, name=None):
        try:
            return importlib.import_module(name or pkg)
        except ImportError:
            print(f"{pkg} not found. Installing...")
            subprocess.run([sys.executable, "-m", "pip", "install", pkg], check=True)
            return importlib.import_module(name or pkg)

    env = {
        "pd": pd,
        "np": import_or_install("numpy"),
        "plt": import_or_install("matplotlib.pyplot", "matplotlib.pyplot"),
        "ad": import_or_install("anndata"),
        "sq": import_or_install("squidpy"),
        "sc": import_or_install("scanpy")
    }
    pd.set_option("display.max_columns", None)
    pd.set_option("display.width", 1200)
    
    print("Environment ready.")
    return env

# Optional: enable cell-level timing for performance awareness
try:
    %load_ext autotime
except ModuleNotFoundError:
    %pip install ipython-autotime
    %load_ext autotime

In [ ]:
env = setup_environment()
pd, np, plt, sns, Image, sc = (
    env["pd"], env["np"], env["plt"], env["ad"],
    env["sq"], env["sc"]
)

## 2. Set Dataset Paths

In [ ]:
HOME     = Path.home()
WS_ROOT  = HOME / "workspace"
WS_FILES = WS_ROOT / "ws_files"

print(f"Home directory : {HOME}")
print(f"Workspace root : {WS_ROOT}")
print(f"Workspace files: {WS_FILES}")

# Mount workspace resources if not already available
if not WS_ROOT.exists():
    print("Workspace not found — attempting to mount...")
    !wb resource mount

WS_FILES.mkdir(parents=True, exist_ok=True)

print("\nWorkspace contents:")
for item in sorted(WS_ROOT.glob("*")):
    print(f"  {'📁' if item.is_dir() else '📄'} {item.name}")

In [ ]:
find_cohort_path("scherzer", "spatial_visium")

---
# 3. Explore Data
---

## Part A: Visium Data

### 1. Locate Visium Dataset

In [ ]:
# Path components — edit these to point to a different dataset
DATASETS_PATH = WS_ROOT / "01_PMDBS" / "pmdbs-spatial-rnaseq-v1"
workflow      = "spatial_visium"
dataset_team  = "scherzer"
dataset_source = "pmdbs"
dataset_type  = "spatial-visium-mtg"
bucket_name   = f"{dataset_team}-{dataset_source}-{dataset_type}"
dataset_name  = f"{dataset_team}-{dataset_source}-{dataset_type}"
visium_path  = DATASETS_PATH / bucket_name / workflow

print("Dataset path:", dataset_path)

### 2. Inspect Available Files

Before loading data it is good practice to check what files are available. The `cohort_analysis/` folder contains the fully processed outputs from the ASAP CRN harmonization pipeline.

In [ ]:
# Build the folder path to the cohort analysis directory
visium_cohort = dataset_path / "cohort_analysis"

# Preview the directory contents
print("Contents of cohort_analysis:")
!ls {visium_cohort}

**Key files to know:**

| File | Description |
|---|---|
| `*.final.h5ad` | Fully processed AnnData — use this for analysis |
| `*.merged_cleaned_unfiltered.h5ad` | Pre-filter AnnData — useful for QC review |
| `*.final_metadata.csv` | Spot-level metadata as a plain CSV |
| `*.sample_list.tsv` | List of samples in this cohort |
| `MANIFEST.tsv` | Provenance and file descriptions |

### 3. Load the processed AnnData

In [ ]:
h5ad_path = visium_cohort / "team-scherzer.final.h5ad"
adata = sc.read_h5ad(h5ad_path)
adata

### 4. Understanding the AnnData Object

Let's unpack what each part of the AnnData object contains.

```
AnnData
├── .X          — gene expression matrix (spots × genes)
├── .obs        — spot-level metadata (one row per spot)
├── .var        — gene-level metadata (one row per gene)
├── .uns        — unstructured data (images, color palettes, etc.)
├── .obsm       — multi-dimensional spot embeddings (PCA, UMAP, spatial coords)
├── .layers     — alternative expression matrices (e.g. raw counts)
└── .obsp       — pairwise distance/connectivity matrices
```

In [ ]:
print("── Dimensions ──────────────────────────────")
print(f"  Spots (observations) : {adata.n_obs:,}")
print(f"  Genes (variables)    : {adata.n_vars:,}")

print("\n── Spot metadata (.obs columns) ─────────────")
print(" ", adata.obs.columns.tolist())

print("\n── Embeddings (.obsm keys) ──────────────────")
print(" ", list(adata.obsm.keys()))
print("  spatial coords shape:", adata.obsm["spatial"].shape)

print("\n── Tissue images (.uns['spatial'] keys) ─────")
print(" ", list(adata.uns["spatial"].keys())[:5], "... (showing first 5)")

print("\n── Layers ───────────────────────────────────")
print(" ", list(adata.layers.keys()))
print("  Tip: .X contains normalized counts; .layers['counts'] contains raw counts")

### 5. Check Spatial Coordinates

Before plotting, always verify that spatial coordinates exist and are sensible. Coordinates are stored in `obsm['spatial']` as a (n_spots × 2) array of pixel positions in the hires image.

In [ ]:
# Assign library_id so squidpy knows which image belongs to which spots
# visium_slide_ref identifies the physical Visium slide (e.g. 'V10D07-371')
adata.obs["library_id"] = adata.obs["visium_slide_ref"]

# Confirm that every library_id has a matching entry in uns['spatial']
ids_match = sorted(adata.obs["library_id"].unique()) == sorted(adata.uns["spatial"].keys())
print(f"Library IDs match uns['spatial'] keys: {ids_match}")

# Spot coordinate ranges — these should fall within image dimensions
print("\nCoordinate ranges per sample (first 5):")
for lib in list(adata.obs["library_id"].unique())[:5]:
    mask   = adata.obs["library_id"] == lib
    coords = adata.obsm["spatial"][mask]
    img    = adata.uns["spatial"][lib]["images"]["hires"]
    print(f"  {lib}: x=[{coords[:,0].min():.0f}, {coords[:,0].max():.0f}]  "
          f"y=[{coords[:,1].min():.0f}, {coords[:,1].max():.0f}]  "
          f"image={img.shape[1]}w × {img.shape[0]}h")

### 6. Visualize — Spatial Scatter

`sq.pl.spatial_scatter` overlays colored spots on the H&E tissue image. Here we color by `leiden` cluster — a pre-computed unsupervised grouping of spots with similar gene expression profiles.

In [ ]:
sq.pl.spatial_scatter(
    adata,
    color="leiden",        # color spots by leiden cluster
    library_key="library_id",  # tells squidpy which column identifies the sample
    figsize=(5, 5),        # size per panel
    title="Leiden clusters",
)

---
## Part B: Loading GeoMx Data

### 1. Locate GeoMx Data

In [ ]:
## Build and set path to desired dataset

DATASETS_PATH = WS_ROOT / "01_PMDBS" / "pmdbs-spatial-rnaseq-v1"

workflow       = "spatial_geomx"
dataset_team   = "edwards"
dataset_source = "pmdbs"
dataset_type   = "spatial-geomx-th"


bucket_name  = f"{dataset_team}-{dataset_source}-{dataset_type}"
dataset_name = f"{dataset_team}-{dataset_source}-{dataset_type}"

geomx_path = DATASETS_PATH / bucket_name / workflow
print("Dataset Path:", dataset_path)


### 2. Inspect Available Files

In [ ]:
# Build the folder path to the cohort analysis directory
cohort_analysis_path = dataset_path / "cohort_analysis"

# Preview the directory contents
print("Contents of cohort_analysis:")
!ls {cohort_analysis_path}

### 3. Load the processed AnnData

In [ ]:
h5ad_path = cohort_analysis_path / "team-edwards.final.h5ad"

adata_geomx = sc.read_h5ad(h5ad_path)
adata_geomx

### 3. Understanding the AnnData Object

In [ ]:
print("── Dimensions ──────────────────────────────")
print(f"  ROIs (observations): {adata_geomx.n_obs}")
print(f"  Genes (variables)  : {adata_geomx.n_vars}")

print("\n── ROI metadata (.obs columns) ─────────────")
print(" ", adata_geomx.obs.columns.tolist())

print("\n── Embeddings (.obsm) ───────────────────────")
print(" ", list(adata_geomx.obsm.keys()))
# Note: no 'spatial' key — GeoMx doesn't have pixel coordinates by default

print("\n── Layers ───────────────────────────────────")
print(" ", list(adata_geomx.layers.keys()))

print("\n── Unique segments (tissue compartments) ────")
print(" ", adata_geomx.obs["segment"].value_counts().to_dict())

print("\n── Unique slides ─────────────────────────────")
print(" ", adata_geomx.obs["slide"].value_counts().to_dict())

In [ ]:
# Preview the first few ROIs
# 'roi' / 'aoi' = region / area of interest identifiers
adata_geomx.obs[["sample", "slide", "segment", "roi", "aoi",
                  "area", "GenesDetected", "GeneDetectionRate", "leiden"]].head(5)

### 4. Visualization

These two plots are standard GeoMx quality checks:

- **Area vs Total Counts:** Larger ROIs should generally have higher counts. If a large ROI has very low counts it may indicate a failed hybridization.
- **Area vs Gene Detection Rate:** The fraction of genes detected above the limit of quantification (LOQ). Values below ~10% may indicate low-quality ROIs.

In [ ]:
# area vs counts and detection rate
fig, axes = plt.subplots(1, 2, figsize=(12,4))
sns.scatterplot(data=adata_geomx.obs, x="area", y="nCount_GeoMx", hue="leiden" if "leiden" in adata_geomx.obs.columns else "slide.name",
                palette="tab20", ax=axes[0], s=60, edgecolor="k", alpha=0.8)
axes[0].set_title("Area vs Total Counts")
sns.scatterplot(data=adata_geomx.obs, x="area", y="GeneDetectionRate", hue="leiden" if "leiden" in adata_geomx.obs.columns else "slide.name",
                palette="tab20", ax=axes[1], s=60, edgecolor="k", alpha=0.8)
axes[1].set_title("Area vs Gene Detection Rate")
for ax in axes:
    ax.legend([],[], frameon=False)
plt.tight_layout()
plt.show()


### 5. UMAP Visualization

Since GeoMx lacks pixel coordinates, UMAP is the primary way to visualize ROI relationships. UMAP projects high-dimensional gene expression into 2D, placing similar ROIs close together.

This dataset already has a precomputed UMAP stored in `obsm['X_umap']` — no recalculation needed.

In [ ]:
# Plot UMAP colored by several biologically relevant variables
# - leiden: unsupervised clusters
# - segment: tissue compartment (most informative for GeoMx)
# - batch: sequencing batch

sc.pl.umap(
    adata_geomx,
    color=["leiden", "segment", "batch"],
    ncols=2,
    frameon=False,
    title=["Leiden clusters", "Tissue segment", "Batch"],
)

---

## 4. Save Outputs

Save the AnnData objects with any modifications you made (e.g. adding `library_id`) so you don't have to repeat these steps in downstream notebooks.

In [ ]:
out_dir = WS_FILES / "spatial_workbook"
out_dir.mkdir(parents=True, exist_ok=True)

visium_out = out_dir / "visium_loaded.h5ad"
geomx_out  = out_dir / "geomx_loaded.h5ad"

adata.write_h5ad(visium_out)
adata_geomx.write_h5ad(geomx_out)

print(f"✓ Visium saved → {visium_out}")
print(f"✓ GeoMx saved  → {geomx_out}")

---

## 5. Next Steps

You have now:
- Loaded a multi-sample Visium AnnData and verified spatial coordinates
- Loaded a GeoMx AnnData and explored ROI-level metadata
- Created basic spatial and UMAP visualizations
- Saved clean Anndata outputs for downstream use

You can now use these outputs as a starting point for your own spatial analysis, including marker exploration, cell type or region comparisons, differential expression, pathway analysis, and biological interpretation.

---

## 6. Provenance

This notebook was developed as part of the **ASAP-CRN Verily Workbench Learning Lab**.

- **Platform:** Verily Workbench
- **Runtime:** JupyterLab (Python 3.12)
- **Key libraries:** `scanpy`, `squidpy`, `anndata`, `pandas`, `numpy`, `matplotlib`
- **Data format:** AnnData (`.h5ad`)
- **Source data:** ASAP CRN post-mortem brain cohort — Spatial RNA-seq (Visium MTG + GeoMx TH)

In [ ]:
!conda env export

In [ ]:
!jupyter labextension list

In [ ]:
!grep ^processor /proc/cpuinfo | wc -l

In [ ]:
!grep "^MemTotal:" /proc/meminfo